# ゲームとエージェントの観測室

画面・操作・DECIDE / RUN・状態の入出力を同じ時点で表示します。**今回以降に記録する走行を、実行中にも保存後にも同じ表示で確認できます。** 上から実行し、記録がまだなければ後半のライブ実行で最初の走行を作ってください。 新しい評価を始めるときだけ、後半の `RUN_LIVE` を `True` にしてください。

- ▶でイベント再生。スライダーや「前／次の操作」で気になる時点へ移動。
- 「最新を追従」をオンにすると約1秒ごとに追記を読み取ります。スライダーを動かすと履歴の位置を固定します。
- 赤丸はクリック位置。選択・送信・受付・次の観測を分けて表示します。
- 状態の入力／出力、実際のHTTP入力とモデル応答、スキル・ツールの呼出しは下のタブ。

表示用の読み取りはゲームを進めません。モデルが出力した記録を表示し、内部思考を推測して補うことはしません。状態は記録された入場・退出に合わせて表示します。


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/notebook_monitor.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.notebook_monitor import AgentMonitor

# 評価ディレクトリ、ゲームの cognition ディレクトリ、または JSONL を指定できます。
# ライブ実行が表示する保存先を指定すれば、終了後にも同じ走行を再生できます。
LOG_ROOT = ROOT / "outputs/evaluations"
print("ログ:", LOG_ROOT)


ログ: /kaggle/working/outputs/evaluations


In [2]:
# このセルの再実行時は、このビューアから開始した評価も停止します。
if "viewer" in globals():
    viewer.close(stop=True)
viewer = AgentMonitor(LOG_ROOT, project=ROOT)
viewer.show()


## 新しい評価をリアルタイムで見る

先にターミナルで `make benchmark-prepare` と `make model-up` を実行してください。JupyterLabは `make lab` の環境を使います。

`RUN_LIVE = True` にして次のセルを実行すると、既存の公式ローカル評価を別プロセスで起動し、上の表示がその出力へ切り替わります。カーネルは操作可能なままです。モデル準備やソース保存中は画面が届くまで少し待ちます。起動エラーは「起動・終了ログ」で確認できます。

`USE_MODEL = False` は可視化・ドライバ確認用の決定的操作です。エージェントの推論能力を評価する条件ではありません。時間・操作予算で終了します。「実行を停止」はこのビューアが起動した評価とその子プロセスだけを止めます。


In [ ]:
RUN_LIVE = False
USE_MODEL = True
GAME = "ls20"
STEPS = 30
SECONDS = 180

if RUN_LIVE:
    output = viewer.start(game=GAME, steps=STEPS, seconds=SECONDS,
                          model=USE_MODEL, learning=True)
    print("新しい記録:", output)
else:
    print("保存済みログの表示のみ。新規評価には RUN_LIVE = True を指定してください。")


## 調査の見方

1. 同じ操作を繰り返す直前で止め、**前後の画面**と**操作の受付**を確認します。
2. DECIDEの入力で課題・直前の結果・仮説が渡っているか、出力で何を更新したかを確認します。
3. RUNの入力と出力を比べ、選んだ操作が実際に送信されたか、スキルの実行や評価が失敗していないかを確認します。
4. スキルを呼ばない場合は「ツール・学習」と「HTTP入力」で、ロードしたスキルと利用可能だったツールを確認します。

追従はローカルログのポーリングです。トークン単位のストリーミングではなく、記録されたイベント単位で更新します。再生速度はゲーム時間とは別です。停止・クラッシュ時に未完了の状態が残ることも、そのまま表示します。

詳しい操作と制約: [可視化ガイド](../docs/agent-monitor-ja.md)
